### Set Explicit Data Directory and Load pyspark

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("data-modeling-data-vault-cust-hist")
    .getOrCreate()
)

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 22:03:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.4


### Simulating a customer attribute change impact on sat_customer

In [4]:
bronze_customers = spark.read.parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

In [5]:
incoming_sat_customer = (
    bronze_customers
    .withColumn(
        "customer_hk",
        F.md5(F.col("customer_id").cast("string"))
    )
    .withColumn(
        "hashdiff",
        F.md5(
            F.concat_ws(
                "||",
                F.coalesce(F.col("first_name"), F.lit("")),
                F.coalesce(F.col("last_name"), F.lit("")),
                F.coalesce(F.col("email"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("state"), F.lit(""))
            )
        )
    )
    .withColumn("load_date", F.current_timestamp())
    .withColumn("record_source", F.lit("crm.customers"))
    .select(
        "customer_hk",
        "hashdiff",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "load_date",
        "record_source"
    )
)

incoming_sat_customer.filter("first_name = 'Alice'").show(truncate=False)

+--------------------------------+--------------------------------+----------+---------+------------------------+---------+-----+-----------------------+-------------+
|customer_hk                     |hashdiff                        |first_name|last_name|email                   |city     |state|load_date              |record_source|
+--------------------------------+--------------------------------+----------+---------+------------------------+---------+-----+-----------------------+-------------+
|b8c37e33defde51cf91e1e03e51657da|03a7e451c8e8dddf2a3cebf5e5328b85|Alice     |Nguyen   |alice.nguyen@example.com|San Diego|CA   |2026-10-04 22:03:26.801|crm.customers|
+--------------------------------+--------------------------------+----------+---------+------------------------+---------+-----+-----------------------+-------------+



In [6]:
existing_sat_customer = spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_customer")
)

In [7]:
new_sat_versions = (
    incoming_sat_customer.alias("incoming")
    .join(
        existing_sat_customer
        .select("customer_hk", "hashdiff")
        .alias("existing"),
        on=[
            F.col("incoming.customer_hk") == F.col("existing.customer_hk"),
            F.col("incoming.hashdiff") == F.col("existing.hashdiff")
        ],
        how="left_anti"
    )
)

new_sat_versions.show(truncate=False)

+--------------------------------+--------------------------------+----------+---------+------------------------+---------+-----+-----------------------+-------------+
|customer_hk                     |hashdiff                        |first_name|last_name|email                   |city     |state|load_date              |record_source|
+--------------------------------+--------------------------------+----------+---------+------------------------+---------+-----+-----------------------+-------------+
|b8c37e33defde51cf91e1e03e51657da|03a7e451c8e8dddf2a3cebf5e5328b85|Alice     |Nguyen   |alice.nguyen@example.com|San Diego|CA   |2026-10-04 22:03:30.797|crm.customers|
+--------------------------------+--------------------------------+----------+---------+------------------------+---------+-----+-----------------------+-------------+



In [8]:
updated_sat_customer = (
    existing_sat_customer
    .unionByName(new_sat_versions)
    .cache()
)

updated_sat_customer.count()

5

In [9]:
updated_sat_customer.write.mode("overwrite").parquet(
    str(DATA_DIR / "data_vault" / "sat_customer")
)

In [10]:
spark.read.parquet(
    str(DATA_DIR / "data_vault" / "sat_customer")
).filter(
    F.col("customer_hk") == "b8c37e33defde51cf91e1e03e51657da"
).orderBy("load_date").show(truncate=False)

+--------------------------------+--------------------------------+----------+---------+------------------------+-----------+-----+-----------------------+-------------+
|customer_hk                     |hashdiff                        |first_name|last_name|email                   |city       |state|load_date              |record_source|
+--------------------------------+--------------------------------+----------+---------+------------------------+-----------+-----+-----------------------+-------------+
|b8c37e33defde51cf91e1e03e51657da|c3b33848b0d75d093a628c78e0c58a5a|Alice     |Nguyen   |alice.nguyen@example.com|Los Angeles|CA   |2026-10-04 21:36:49.506|crm.customers|
|b8c37e33defde51cf91e1e03e51657da|03a7e451c8e8dddf2a3cebf5e5328b85|Alice     |Nguyen   |alice.nguyen@example.com|San Diego  |CA   |2026-10-04 22:03:31.409|crm.customers|
+--------------------------------+--------------------------------+----------+---------+------------------------+-----------+-----+-------------------